# Lab 02 - Generadores pseudoaleatorios y transformada inversa

**Modelación y Simulación**  
En este notebook se implementan los cinco ejercicios del laboratorio. Se usa $\alpha=0.05$ en todas las pruebas de hipótesis.

In [ ]:
import gc
import time
from math import ceil, log

import matplotlib.pyplot as plt
import nistrng
import numpy as np
from scipy import stats

ALPHA = 0.05
np.set_printoptions(precision=4, suppress=True)

## Funciones auxiliares

La regla de decisión usada es: no se rechaza $H_0$ cuando el p-value es mayor o igual a $\alpha$.

In [ ]:
def conclusion(p_value, alpha=ALPHA):
    return "No se rechaza H0" if p_value >= alpha else "Se rechaza H0"


def describe_uniform(sample, label, bins=20):
    ks = stats.kstest(sample, "uniform")
    counts, _ = np.histogram(sample, bins=bins, range=(0, 1))
    chi = stats.chisquare(counts)
    print(f"{label}: n={len(sample)}, media={sample.mean():.5f}, varianza={sample.var(ddof=1):.5f}")
    print(f"  KS: estadístico={ks.statistic:.5f}, p-value={ks.pvalue:.5g}. {conclusion(ks.pvalue)}")
    print(f"  Chi-cuadrado: estadístico={chi.statistic:.5f}, p-value={chi.pvalue:.5g}. {conclusion(chi.pvalue)}")
    plt.figure(figsize=(7, 3.5))
    plt.hist(sample, bins=bins, range=(0, 1), density=True, edgecolor="black", alpha=0.75)
    plt.axhline(1, color="red", linestyle="--", label="densidad teórica")
    plt.title(label)
    plt.xlabel("valor")
    plt.ylabel("densidad")
    plt.legend()
    plt.show()

# 1. Generador congruencial lineal (LCG)

Se genera primero la muestra discreta $x_1,\ldots,x_N$ y luego se normaliza como $u_i=x_i/m$ para obtener valores en $[0,1)$.

In [ ]:
def lcg(seed, a, c, m, n):
    values = np.empty(n, dtype=np.int64)
    x = seed
    for i in range(n):
        x = (a * x + c) % m
        values[i] = x
    return values


# Comprobaciones pequeñas antes de los experimentos.
check = lcg(1, 5, 1, 16, 10)
assert len(check) == 10
assert np.all((check >= 0) & (check < 16))

lcg_cases = [
    {"name": "LCG 1: Numerical Recipes", "seed": 12345, "a": 1664525, "c": 1013904223, "m": 2**32},
    {"name": "LCG 2: Park-Miller", "seed": 12345, "a": 16807, "c": 0, "m": 2**31 - 1},
]
N_UNIFORM = 10_000
lcg_uniform_samples = {}

for params in lcg_cases:
    discrete = lcg(params["seed"], params["a"], params["c"], params["m"], N_UNIFORM)
    uniform = discrete / params["m"]
    lcg_uniform_samples[params["name"]] = uniform
    print(f"\n{params['name']}: a={params['a']}, c={params['c']}, m={params['m']}, N={N_UNIFORM}")
    print("Primeros 10 valores discretos:", discrete[:10])
    describe_uniform(uniform, params["name"])

# 2. Mersenne Twister

Se implementa MT19937 con el tamaño de estado estándar de 624 enteros de 32 bits. Para el experimento se convierte cada entero a un valor uniforme dividiendo entre $2^{32}$.

In [ ]:
class MersenneTwister:
    def __init__(self, seed=5489):
        self.n, self.m = 624, 397
        self.matrix_a = 0x9908B0DF
        self.upper_mask = 0x80000000
        self.lower_mask = 0x7FFFFFFF
        self.state = np.zeros(self.n, dtype=np.uint32)
        self.state[0] = seed & 0xFFFFFFFF
        for i in range(1, self.n):
            self.state[i] = (1812433253 * (int(self.state[i - 1]) ^ (int(self.state[i - 1]) >> 30)) + i) & 0xFFFFFFFF
        self.index = self.n

    def twist(self):
        for i in range(self.n):
            x = (int(self.state[i]) & self.upper_mask) + (int(self.state[(i + 1) % self.n]) & self.lower_mask)
            x_a = x >> 1
            if x & 1:
                x_a ^= self.matrix_a
            self.state[i] = int(self.state[(i + self.m) % self.n]) ^ x_a
        self.index = 0

    def random_uint32(self):
        if self.index >= self.n:
            self.twist()
        y = int(self.state[self.index])
        y ^= y >> 11
        y ^= (y << 7) & 0x9D2C5680
        y ^= (y << 15) & 0xEFC60000
        y ^= y >> 18
        self.index += 1
        return y & 0xFFFFFFFF

    def random(self):
        return self.random_uint32() / 2**32


mt_check = MersenneTwister(7)
assert 0 <= mt_check.random() < 1

mt = MersenneTwister(2026)
mt_uniform = np.fromiter((mt.random() for _ in range(N_UNIFORM)), dtype=float, count=N_UNIFORM)
describe_uniform(mt_uniform, "Mersenne Twister MT19937")